In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# used to remove the old processed dataset if performing the split again
import shutil

shutil.rmtree(
    "/content/drive/MyDrive/SIH_Crop_Project/potato_dataset/data_potato_processesd_3",
    ignore_errors=True
)

In [8]:
import os
import random
import shutil
from pathlib import Path

# =========================
# CONFIG
# =========================

SOURCE_DIR = "/content/drive/MyDrive/SIH_Crop_Project/potato_dataset/potato_data_2"

OUTPUT_DIR = "/content/drive/MyDrive/SIH_Crop_Project/potato_dataset/data_potato_processesd_3"

TRAIN_SPLIT = 0.80
VAL_SPLIT = 0.15
TEST_SPLIT = 0.5

SEED = 42

# =========================
# REPRODUCIBILITY
# =========================

random.seed(SEED)

# =========================
# CREATE OUTPUT FOLDERS
# =========================

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

# =========================
# PROCESS EACH CLASS
# =========================

classes = os.listdir(SOURCE_DIR)

for class_name in classes:

    class_path = os.path.join(SOURCE_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)

    # Shuffle images randomly
    random.shuffle(images)

    total_images = len(images)

    # Split indices
    train_end = int(total_images * TRAIN_SPLIT)
    val_end = train_end + int(total_images * VAL_SPLIT)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    split_map = {
        "train": train_images,
        "val": val_images,
        "test": test_images
    }

    # =========================
    # COPY FILES
    # =========================

    for split_name, split_images in split_map.items():

        split_class_dir = os.path.join(
            OUTPUT_DIR,
            split_name,
            class_name
        )

        os.makedirs(split_class_dir, exist_ok=True)

        for image_name in split_images:

            src = os.path.join(class_path, image_name)

            dst = os.path.join(split_class_dir, image_name)

            shutil.copy2(src, dst)

    # =========================
    # PRINT STATS
    # =========================

    print(f"\nClass: {class_name}")
    print(f"Total: {total_images}")
    print(f"Train: {len(train_images)}")
    print(f"Val: {len(val_images)}")
    print(f"Test: {len(test_images)}")

print("\nDataset split completed successfully.")


Class: Potato___Early_blight
Total: 1048
Train: 838
Val: 157
Test: 53

Class: Potato___healthy
Total: 1000
Train: 800
Val: 150
Test: 50

Class: Potato___Late_blight
Total: 1126
Train: 900
Val: 168
Test: 58

Dataset split completed successfully.


In [9]:
import os

DATASET_PATH = "/content/drive/MyDrive/SIH_Crop_Project/potato_dataset/data_potato_processesd_3"

splits = ["train", "val", "test"]

grand_total = 0

for split in splits:

    split_path = os.path.join(DATASET_PATH, split)

    total_images = 0

    print(f"\n===== {split.upper()} =====")

    classes = os.listdir(split_path)

    for class_name in classes:

        class_path = os.path.join(split_path, class_name)

        num_images = len(os.listdir(class_path))

        total_images += num_images

        print(f"{class_name}: {num_images}")

    grand_total += total_images

    print(f"\nTotal {split} images: {total_images}")

print(f"\nTOTAL DATASET IMAGES: {grand_total}")


===== TRAIN =====
Potato___Early_blight: 838
Potato___healthy: 800
Potato___Late_blight: 900

Total train images: 2538

===== VAL =====
Potato___Early_blight: 157
Potato___healthy: 150
Potato___Late_blight: 168

Total val images: 475

===== TEST =====
Potato___Early_blight: 53
Potato___healthy: 50
Potato___Late_blight: 58

Total test images: 161

TOTAL DATASET IMAGES: 3174


In [10]:
#corrupted image detection
from PIL import Image
import os

DATASET_PATH = "/content/drive/MyDrive/SIH_Crop_Project/potato_dataset/data_potato_processesd_3"

corrupted_images = []

splits = ["train", "val", "test"]

for split in splits:

    split_path = os.path.join(DATASET_PATH, split)

    classes = os.listdir(split_path)

    print(f"\nChecking {split} set...")

    for class_name in classes:

        class_path = os.path.join(split_path, class_name)

        images = os.listdir(class_path)

        for image_name in images:

            image_path = os.path.join(class_path, image_name)

            try:
                with Image.open(image_path) as img:
                    img.verify()

            except Exception:

                corrupted_images.append(image_path)

                print(f"Corrupted: {image_path}")

print("\n===== FINAL REPORT =====")

print(f"Total corrupted images: {len(corrupted_images)}")


Checking train set...

Checking val set...

Checking test set...

===== FINAL REPORT =====
Total corrupted images: 0
